In [0]:
%pip install catboost openpyxl

In [0]:
from pathlib import Path
import pandas as pd

# Datenordner aus deinem aktuellen NMF-Notebook
basis = Path("/Workspace/Users/khalil-said.albert@de.abb.com/Daten")

# Passende Excel-Dateien suchen
bom_dateien = sorted(basis.glob("BIG2_CORR_191007_Stuecklisten.aufgeloest (002)_korrigiert.xlsx"))

# Benötigte Dateien prüfen
datei_check = pd.DataFrame([
    {"Prüfung": "bom_core.py vorhanden",
     "OK": (basis / "bom_core.py").is_file()},
    {"Prüfung": "layer2_profile.py vorhanden",
     "OK": (basis / "layer2_profile.py").is_file()},
    {"Prüfung": "Genau eine korrigierte Stückliste vorhanden",
     "OK": len(bom_dateien) == 1},
])

display(datei_check)

In [0]:
import sys
import time

assert datei_check["OK"].all(), "Bitte zuerst die fehlenden Dateien prüfen."

# Python soll eure Module im Datenordner finden
if str(basis) not in sys.path:
    sys.path.insert(0, str(basis))

import bom_core as bc
import layer2_profile as L2

# Die zuvor gefundene Stückliste mit dem bestehenden Cleaner laden
arbeitsdatei = str(bom_dateien[0])
start = time.perf_counter()

sauber, protokoll = bc.clean_bom(arbeitsdatei)

display(pd.DataFrame([{
    "Positionszeilen": len(sauber),
    "Erwartete Positionszeilen": 47390,
    "Stand stimmt überein": len(sauber) == 47390,
    "Laufzeit in Sekunden": round(time.perf_counter() - start, 1)
}]))

In [0]:
# Aus der Liste eine Tabelle mit den drei benötigten Spalten bilden
positionen = pd.DataFrame(sauber)[
    ["erzeugnis", "block", "komponente"]
].copy()

assert positionen.notna().all().all(), "Produkt, Block oder Komponente fehlt."

# Pro Produktblock die verschiedenen Komponenten zusammenstellen
block_sets = (
    positionen.groupby(["erzeugnis", "block"])["komponente"]
    .agg(lambda komponenten: tuple(sorted(set(komponenten))))
    .reset_index(name="komponentenset")
)

# Pro Produkt die Blöcke und unterschiedlichen Komponentensets zählen
block_check = (
    block_sets.groupby("erzeugnis")
    .agg(
        bloecke=("block", "size"),
        unterschiedliche_sets=("komponentenset", "nunique")
    )
)

display(pd.DataFrame([{
    "Produkte": len(block_check),
    "Produktblöcke": len(block_sets),
    "Produkte mit mehreren Blöcken": int((block_check["bloecke"] > 1).sum()),
    "Davon mit unterschiedlichen Komponentensets":
        int((block_check["unterschiedliche_sets"] > 1).sum())
}]))

In [0]:
# Gleiche Komponentensets innerhalb desselben Produkts einmal zählen
fassungen = (
    block_sets[["erzeugnis", "komponentenset"]]
    .drop_duplicates(["erzeugnis", "komponentenset"])
    .reset_index(drop=True)
)

# Jede unterschiedliche Fassung bekommt eine technische Kennung
fassungen["fassung_id"] = fassungen.index

# Alle Fassungen eines Produkts erhalten zusammen das Gewicht 1
anzahl_fassungen = (
    fassungen.groupby("erzeugnis")["fassung_id"].transform("count")
)
fassungen["produktgewicht"] = 1.0 / anzahl_fassungen

display(pd.DataFrame([{
    "Produkte": fassungen["erzeugnis"].nunique(),
    "Unterschiedliche Fassungen": len(fassungen),
    "Produkte mit mehreren Fassungen":
        fassungen.loc[anzahl_fassungen > 1, "erzeugnis"].nunique(),
    "Summe der Produktgewichte": fassungen["produktgewicht"].sum()
}]))

In [0]:
# Hauptgruppe aus der Erzeugnisnummer ableiten (erste 6 Zeichen)
fassungen_mit_gruppen = fassungen.assign(
    hauptgruppe=fassungen["erzeugnis"].apply(L2.hauptgruppe_von)
)

# Jede Fassung muss einer Hauptgruppe zugeordnet sein
assert fassungen_mit_gruppen["hauptgruppe"].notna().all()

X_je_hauptgruppe = {}
info_je_hauptgruppe = {}

for hauptgruppe, gruppe in fassungen_mit_gruppen.groupby("hauptgruppe"):
    # Aus jedem Komponentenset eine Zeile je enthaltener Komponente machen
    vorhanden = (
        gruppe[["fassung_id", "komponentenset"]]
        .explode("komponentenset")
        .rename(columns={"komponentenset": "komponente"})
    )

    # Anwesenheit als 0/1-Tabelle darstellen
    X = pd.crosstab(vorhanden["fassung_id"], vorhanden["komponente"])
    X = X.reindex(index=gruppe["fassung_id"], fill_value=0)
    X = (X > 0).astype("int8")

    X_je_hauptgruppe[hauptgruppe] = X
    info_je_hauptgruppe[hauptgruppe] = (
        gruppe.set_index("fassung_id").loc[X.index].copy()
    )

display(pd.DataFrame([
    {
        "Hauptgruppe": hg,
        "Produkte": info_je_hauptgruppe[hg]["erzeugnis"].nunique(),
        "Fassungen": X.shape[0],
        "Verschiedene Komponenten": X.shape[1]
    }
    for hg, X in X_je_hauptgruppe.items()
]))

In [0]:
import numpy as np
import pandas as pd

HAUPTGRUPPEN_BIG2 = ("GJL121", "GJL131")
SEED = 42
N_FOLDS = 5

info_pilot = pd.concat(
    [
        info_je_hauptgruppe[hg].assign(hauptgruppe=hg)
        for hg in HAUPTGRUPPEN_BIG2
    ]
).copy()

info_pilot["erzeugnis"] = info_pilot["erzeugnis"].astype(str)

assert info_pilot.index.is_unique
assert (
    info_pilot.groupby("erzeugnis")["hauptgruppe"]
    .nunique().eq(1).all()
)

fold_je_produkt = {}

for gruppen_nummer, hg in enumerate(HAUPTGRUPPEN_BIG2):
    produkte = np.array(
        sorted(
            info_pilot.loc[
                info_pilot["hauptgruppe"].eq(hg), "erzeugnis"
            ].unique()
        ),
        dtype=object,
    )

    np.random.default_rng(
        SEED + gruppen_nummer
    ).shuffle(produkte)

    fold_je_produkt.update({
        str(produkt): nummer % N_FOLDS
        for nummer, produkt in enumerate(produkte)
    })

info_pilot["fold"] = (
    info_pilot["erzeugnis"].map(fold_je_produkt)
)

assert info_pilot["fold"].notna().all()
assert (
    info_pilot.groupby("erzeugnis")["fold"]
    .nunique().eq(1).all()
)

display(
    info_pilot.groupby(
        ["hauptgruppe", "fold"]
    ).agg(
        Produkte=("erzeugnis", "nunique"),
        Fassungen=("erzeugnis", "size"),
    ).reset_index()
)

In [0]:
# BIG2 – Zelle 1: Tatsächliche Mengen- und Positionszustände vermessen

import numpy as np
import pandas as pd


big2_quelle = pd.DataFrame(sauber).copy()

pflichtfelder = [
    "erzeugnis",
    "block",
    "komponente",
    "menge",
    "tiefe",
    "alt_gruppe",
    "alt_wahrsch",
]

fehlende_felder = [
    feld for feld in pflichtfelder
    if feld not in big2_quelle.columns
]

assert not fehlende_felder, (
    f"Für Big2 fehlen diese Felder: {fehlende_felder}"
)


# Hauptgruppe und Tiefe sauber aufbereiten
big2_quelle["hauptgruppe"] = (
    big2_quelle["erzeugnis"]
    .apply(L2.hauptgruppe_von)
)

big2_quelle["tiefe_num"] = pd.to_numeric(
    big2_quelle["tiefe"],
    errors="coerce"
)


# Mengen robust in Zahlen umwandeln
def menge_als_zahl(wert):
    if pd.isna(wert):
        return np.nan

    if isinstance(wert, (int, float, np.integer, np.floating)):
        return float(wert)

    text = str(wert).strip().replace(" ", "")

    # Deutsches Dezimalkomma auffangen
    if "," in text and "." not in text:
        text = text.replace(",", ".")

    return pd.to_numeric(text, errors="coerce")


big2_quelle["menge_num"] = (
    big2_quelle["menge"]
    .map(menge_als_zahl)
)


# Alternativpositionen NICHT ausschließen
alt_text = (
    big2_quelle["alt_gruppe"]
    .fillna("")
    .astype(str)
    .str.strip()
)

big2_quelle["ist_alternative"] = (
    alt_text.ne("")
    & alt_text.ne("0")
    & alt_text.str.upper().ne("NAN")
)


# Konservativer erster Big2-Umfang:
# Hauptgruppe GJL121 und Tiefe 1.
# Menge 0 und Alternativpositionen bleiben ausdrücklich enthalten.
big2_positionen = big2_quelle.loc[
    big2_quelle["hauptgruppe"].isin(HAUPTGRUPPEN_BIG2)
    & big2_quelle["tiefe_num"].eq(1)
].copy()

assert not big2_positionen.empty
assert big2_positionen["erzeugnis"].isin(fold_je_produkt).all()

big2_positionen["fold"] = (
    big2_positionen["erzeugnis"]
    .map(fold_je_produkt)
)


# Mengen lesbar und stabil darstellen
def mengenwert_als_text(wert):
    if pd.isna(wert):
        return "FEHLT"

    return f"{float(wert):.12g}"


def mengenmuster(werte):
    werte = sorted(
        float(wert)
        for wert in werte
        if pd.notna(wert)
    )

    if not werte:
        return "FEHLT"

    return "|".join(
        mengenwert_als_text(wert)
        for wert in werte
    )


# Ein vorhandener Big2-Fall:
# Produkt + Block + exakte Komponente
big2_faelle = (
    big2_positionen
    .groupby(
        ["erzeugnis", "block", "komponente"],
        observed=True
    )
    .agg(
        positionsanzahl=("komponente", "size"),
        mengenmuster=("menge_num", mengenmuster),
        fehlende_mengenwerte=(
            "menge_num",
            lambda werte: int(werte.isna().sum())
        ),
        alternativpositionen=(
            "ist_alternative",
            "sum"
        ),
        fold=("fold", "first"),
    )
    .reset_index()
)


big2_faelle["zustand_vorhanden"] = (
    "POSITIONEN_"
    + big2_faelle["positionsanzahl"].astype(str)
    + "__MENGEN_"
    + big2_faelle["mengenmuster"]
)


# Vollständige Big2-Grundgesamtheit:
# Jeder Block × jede auf Tiefe 1 bekannte Komponente
bloecke = (
    big2_positionen[
        ["erzeugnis", "block", "fold"]
    ]
    .drop_duplicates()
)

komponenten = pd.DataFrame({
    "komponente": sorted(
        big2_positionen["komponente"].unique()
    )
})

big2_universum = bloecke.merge(
    komponenten,
    how="cross"
)

big2_universum = big2_universum.merge(
    big2_faelle[
        [
            "erzeugnis",
            "block",
            "komponente",
            "positionsanzahl",
            "mengenmuster",
            "fehlende_mengenwerte",
            "alternativpositionen",
            "zustand_vorhanden",
        ]
    ],
    on=["erzeugnis", "block", "komponente"],
    how="left",
    validate="one_to_one",
)

big2_universum["zustand_big2"] = (
    big2_universum["zustand_vorhanden"]
    .fillna("NICHT_GELISTET")
)

big2_universum["vorhanden_big2"] = (
    big2_universum["zustand_big2"]
    .ne("NICHT_GELISTET")
    .astype("int8")
)


# Mehrfachblöcke desselben Produkts untersuchen
blockanzahl = (
    bloecke.groupby("erzeugnis")["block"]
    .nunique()
)

mehrblock_produkte = blockanzahl[
    blockanzahl > 1
].index

blockabweichungen = (
    big2_universum.loc[
        big2_universum["erzeugnis"].isin(
            mehrblock_produkte
        )
    ]
    .groupby(
        ["erzeugnis", "komponente"]
    )["zustand_big2"]
    .nunique()
)

anzahl_blockabweichungen = int(
    blockabweichungen.gt(1).sum()
)


# Übersicht
display(pd.DataFrame([{
    "Produkte": big2_positionen["erzeugnis"].nunique(),
    "Produktblöcke": bloecke.shape[0],
    "Komponenten auf Tiefe 1":
        big2_positionen["komponente"].nunique(),
    "Positionszeilen": len(big2_positionen),
    "Big2-Fälle vorhanden": len(big2_faelle),
    "Big2-Paare vollständig": len(big2_universum),
    "Verschiedene Mengenwerte":
        big2_positionen["menge_num"].nunique(dropna=True),
    "Positionen mit Menge 0":
        int(big2_positionen["menge_num"].eq(0).sum()),
    "Positionen mit negativer Menge":
        int(big2_positionen["menge_num"].lt(0).sum()),
    "Positionen ohne lesbare Menge":
        int(big2_positionen["menge_num"].isna().sum()),
    "Alternativpositionen":
        int(big2_positionen["ist_alternative"].sum()),
    "Mehrblock-Produkte": len(mehrblock_produkte),
    "Produkt-Komponenten mit unterschiedlichen Blockzuständen":
        anzahl_blockabweichungen,
}]))


print("Häufigste einzelne Mengenwerte:")

display(
    big2_positionen
    .groupby("menge_num", dropna=False)
    .agg(
        Positionszeilen=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
        Komponenten=("komponente", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Positionszeilen",
        ascending=False
    )
    .head(30)
)


print("Häufigste vollständige Big2-Zustände:")

display(
    big2_universum
    .groupby("zustand_big2")
    .agg(
        Prüfpaare=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
        Komponenten=("komponente", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Prüfpaare",
        ascending=False
    )
    .head(30)
)


print("Alternativinformationen im Big2-Umfang:")

display(
    big2_positionen.assign(
        alt_wahrsch_anzeige=(
            big2_positionen["alt_wahrsch"]
            .fillna("OHNE_ANGABE")
            .astype(str)
        )
    )
    .groupby(
        ["ist_alternative", "alt_wahrsch_anzeige"],
        dropna=False
    )
    .agg(
        Positionszeilen=("komponente", "size"),
        Produkte=("erzeugnis", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Positionszeilen",
        ascending=False
    )
)

In [0]:
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier


positionen_b2 = pd.DataFrame(big2_positionen).copy()
pflicht = {"erzeugnis", "block", "komponente", "menge_num"}
fehlende_spalten = pflicht - set(positionen_b2.columns)
assert not fehlende_spalten, f"Big2-Eingabe ohne Spalten: {fehlende_spalten}"
assert positionen_b2[list(pflicht)].notna().all().all(), (
    "Produkt, Block, Komponente oder Menge fehlt: zuerst die Vorbereitung prüfen."
)
positionen_b2["erzeugnis"] = positionen_b2["erzeugnis"].astype(str)
positionen_b2["komponente"] = positionen_b2["komponente"].astype(str)
positionen_b2["menge_num"] = pd.to_numeric(
    positionen_b2["menge_num"], errors="raise"
).astype(float)
assert np.isfinite(positionen_b2["menge_num"]).all()

# Mengen/Positionen derselben Materialnummer in einem Produktblock bleiben zusammen.
faelle_b2 = (
    positionen_b2.groupby(["erzeugnis", "block", "komponente"], sort=False)["menge_num"]
    .agg(lambda werte: tuple(sorted(werte)))
    .reset_index(name="mengen_je_position")
)
faelle_b2["anzahl_positionen"] = faelle_b2["mengen_je_position"].map(len)
faelle_b2["menge_summe"] = faelle_b2["mengen_je_position"].map(sum)
# Ziel: Gesamtmenge UND Anzahl gleicher Materialpositionen.
# Gemischte Einzelmengen sind erlaubt; ihre Aufteilung ist kein eigenes Lernziel.
def menge_key_b2(q):
    return format(float(q), ".12g")

faelle_b2["menge_key"] = faelle_b2["menge_summe"].map(menge_key_b2)
faelle_b2["zustand_big2"] = (
    faelle_b2["menge_key"] + "__" + faelle_b2["anzahl_positionen"].astype(str)
)
assert faelle_b2["anzahl_positionen"].sum() == len(positionen_b2)

block_index = pd.MultiIndex.from_frame(
    faelle_b2[["erzeugnis", "block"]].drop_duplicates()
)
komponenten_b2 = sorted(faelle_b2["komponente"].unique())

positionen_matrix = (
    faelle_b2.pivot(index=["erzeugnis", "block"], columns="komponente",
                    values="anzahl_positionen")
    .reindex(index=block_index, columns=komponenten_b2)
    .fillna(0).to_numpy(dtype=np.float32)
)
mengen_matrix = (
    faelle_b2.pivot(index=["erzeugnis", "block"], columns="komponente",
                    values="menge_summe")
    .reindex(index=block_index, columns=komponenten_b2)
    .fillna(0).to_numpy(dtype=np.float32)
)

block_je_fall = block_index.get_indexer(
    pd.MultiIndex.from_frame(faelle_b2[["erzeugnis", "block"]])
)
spalte_je_fall = pd.Index(komponenten_b2).get_indexer(faelle_b2["komponente"])
assert (block_je_fall >= 0).all() and (spalte_je_fall >= 0).all()
zeilen_b2 = np.arange(len(faelle_b2))

# Gleicher Mechanismus wie Big1: eigene Antwort verdecken, andere Komponenten sehen.
kontext_positionen = positionen_matrix[block_je_fall].copy()
kontext_mengen = mengen_matrix[block_je_fall].copy()
assert np.array_equal(
    kontext_positionen[zeilen_b2, spalte_je_fall],
    faelle_b2["anzahl_positionen"].to_numpy(dtype=np.float32)
)
assert np.allclose(
    kontext_mengen[zeilen_b2, spalte_je_fall],
    faelle_b2["menge_summe"].to_numpy(dtype=np.float32)
)
kontext_positionen[zeilen_b2, spalte_je_fall] = 0
kontext_mengen[zeilen_b2, spalte_je_fall] = 0

features_big2_neu = pd.DataFrame(
    np.concatenate([kontext_positionen, kontext_mengen], axis=1),
    columns=([f"anzahl_kontext_{i}" for i in range(len(komponenten_b2))]
             + [f"menge_kontext_{i}" for i in range(len(komponenten_b2))]),
)
features_big2_neu["pruefkomponente"] = faelle_b2["komponente"].to_numpy()
assert (kontext_positionen[zeilen_b2, spalte_je_fall] == 0).all()
assert (kontext_mengen[zeilen_b2, spalte_je_fall] == 0).all()

fold_map_b2 = {str(produkt): fold for produkt, fold in fold_je_produkt.items()}
faelle_b2["fold"] = faelle_b2["erzeugnis"].map(fold_map_b2)
assert faelle_b2["fold"].notna().all(), "Big2-Produkt ohne Big1-Fold."
assert faelle_b2.groupby("erzeugnis")["fold"].nunique().eq(1).all()

blockanzahl_b2 = (
    faelle_b2[["erzeugnis", "block"]].drop_duplicates()
    .groupby("erzeugnis").size()
)
faelle_b2["blockgewicht"] = 1.0 / faelle_b2["erzeugnis"].map(blockanzahl_b2)


print(f"Big2: {len(faelle_b2)} vorhandene Fälle; {features_big2_neu.shape[1]} Eingaben.")
print("Zielgröße: Gesamtmenge und Positionszahl. Eigene Menge und Positionszahl verdeckt.")

In [0]:
def auswerten_big2(faelle, wahrscheinlichkeiten, klasseninfo, trainingskomponenten):
    """Marginale Mengenverteilung; Positionszahl bedingt auf die Ist-Menge.

    Keine fachlichen Sollmengen, Mindesthäufigkeiten oder Fehlerschwellen.
    Gleichstand bedeutet keine bevorzugte Gegenhypothese.
    Score = 1-p(Ist), keine Wahrscheinlichkeit eines fachlichen Fehlers.
    """
    p = np.asarray(wahrscheinlichkeiten, dtype=float)
    assert p.shape == (len(faelle), len(klasseninfo))
    assert np.allclose(p.sum(axis=1), 1)
    q_keys = list(dict.fromkeys(klasseninfo["menge_key"]))
    q_mask = np.array([klasseninfo["menge_key"].eq(q).to_numpy() for q in q_keys])
    pq = p @ q_mask.T
    ergebnisse = []
    for i, (_, fall) in enumerate(faelle.iterrows()):
        ist_q = menge_key_b2(fall["menge_summe"])
        ist_n = int(fall["anzahl_positionen"])
        best_q_index = int(pq[i].argmax())
        best_q_key = q_keys[best_q_index]
        p_best = float(pq[i, best_q_index])
        p_ist = float(pq[i, q_keys.index(ist_q)]) if ist_q in q_keys else np.nan
        if np.isfinite(p_ist) and abs(p_best - p_ist) <= 1e-12:
            best_q_key = ist_q
        gesehen = fall["komponente"] in trainingskomponenten
        grund, kandidat, score = "", False, np.nan
        status = "bewertbar"
        expected_n, p_n_ist, p_n_best = np.nan, np.nan, np.nan
        # Auch die Positionsprüfung summiert Zustandswahrscheinlichkeiten,
        # aber nur innerhalb derselben Gesamtmenge.
        if ist_q in q_keys:
            mask = klasseninfo["menge_key"].eq(ist_q).to_numpy()
            n_values = sorted(klasseninfo.loc[mask, "anzahl_positionen"].unique())
            pn = np.array([p[i, mask & klasseninfo["anzahl_positionen"].eq(v).to_numpy()].sum()
                           for v in n_values])
            pn = pn / pn.sum() if pn.sum() > 0 else np.zeros_like(pn)
            expected_n = int(n_values[int(pn.argmax())])
            p_n_best = float(pn.max())
            if ist_n in n_values:
                p_n_ist = float(pn[n_values.index(ist_n)])
        if not gesehen:
            status, grund = "komponente_nicht_im_training", "Komponente im Training unbekannt"
        elif not np.isfinite(p_ist):
            status, grund, kandidat = "menge_nicht_im_training", "Menge im Training unbekannt", True
        elif best_q_key != ist_q and p_best > p_ist + 1e-12:
            grund, kandidat, score = "Menge abweichend", True, 1 - p_ist
        elif not np.isfinite(p_n_ist):
            status, kandidat = "positionszahl_bei_menge_unbekannt", True
            grund = ("möglicherweise doppelte Position" if ist_n > expected_n
                     else "Positionszahl im Training unbekannt")
        elif expected_n != ist_n and p_n_best > p_n_ist + 1e-12:
            grund = ("möglicherweise doppelte Position" if ist_n > expected_n
                     else "Positionszahl abweichend")
            kandidat, score = True, 1 - p_n_ist
        ergebnisse.append({
            "beobachtete_menge": float(fall["menge_summe"]),
            "erwartete_menge": float(best_q_key) if gesehen else np.nan,
            "erwartete_positionszahl_bei_ist_menge": expected_n if gesehen else np.nan,
            "p_ist_menge": p_ist if gesehen else np.nan,
            "p_erwartete_menge": p_best if gesehen else np.nan,
            "p_ist_positionszahl_bedingt": p_n_ist if gesehen else np.nan,
            "kandidat_big2": kandidat, "grund_big2": grund,
            "auffaelligkeit_big2": score, "pruefstatus_big2": status,
        })
    return pd.concat([faelle.reset_index(drop=True), pd.DataFrame(ergebnisse)], axis=1)


def prognose_big2(modell, train_faelle, test_faelle, test_features):
    info = train_faelle.drop_duplicates("zustand_big2").set_index("zustand_big2")
    info = info.loc[np.asarray(modell.classes_).astype(str)]
    return auswerten_big2(test_faelle, modell.predict_proba(test_features), info,
                         set(train_faelle["komponente"]))

In [0]:
import re
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

FEHLT_BIG2 = "OHNE_ANGABE"

# Gleiche Merkmalsbildung wie bei Big1.
def spannung_big2(text):
    text = str(text).replace("*", " ").replace("+", " ").replace("_", " ")
    treffer = re.findall(r"(\d+(?:[-/]\d+)?)\s*V\s*(DC|AC)?", text, re.I)
    werte = set()

    for wert, art in treffer:
        art = art.upper() or (
            "AC" if re.search(r"\d+(?:-\d+)?\s*HZ", text, re.I)
            else "UNBEKANNT"
        )
        werte.add(f"{wert}_{art}")

    return (
        next(iter(werte)) if len(werte) == 1
        else "MEHRDEUTIG" if werte
        else FEHLT_BIG2
    )


def eindeutig_big2(werte):
    werte = set(werte.fillna(FEHLT_BIG2))
    return next(iter(werte)) if len(werte) == 1 else "MEHRDEUTIG"


quelle = pd.DataFrame(sauber)[[
    "erzeugnis", "erzeugnis_txt", "komponente", "komponente_txt"
]].copy()

quelle["erzeugnis"] = quelle["erzeugnis"].astype(str)
quelle["komponente"] = quelle["komponente"].astype(str)
quelle = quelle.loc[
    quelle["erzeugnis"].isin(faelle_b2["erzeugnis"])
].copy()

quelle["produkt_spannung"] = quelle["erzeugnis_txt"].map(spannung_big2)
quelle["komponente_spannung"] = quelle["komponente_txt"].map(spannung_big2)

produkt_spannung = quelle.groupby("erzeugnis")[
    "produkt_spannung"
].agg(eindeutig_big2)

komponente_spannung = quelle.groupby("komponente")[
    "komponente_spannung"
].agg(eindeutig_big2)

# Numerischer Stücklistenkontext und Maskierung bleiben erhalten.
features_big2_neu = features_big2_neu.copy()
assert features_big2_neu.index.equals(faelle_b2.index)
assert np.array_equal(
    features_big2_neu["pruefkomponente"].astype(str).to_numpy(),
    faelle_b2["komponente"].astype(str).to_numpy(),
)

features_big2_neu["produkt_spannung"] = (
    faelle_b2["erzeugnis"].map(produkt_spannung).fillna(FEHLT_BIG2)
)
features_big2_neu["komponente_spannung"] = (
    faelle_b2["komponente"].map(komponente_spannung).fillna(FEHLT_BIG2)
)
features_big2_neu["komponente_basis"] = (
    faelle_b2["komponente"].astype(str)
    .str.extract(r"^(.+?)[A-Za-z]\d{4}$", expand=False)
    .fillna(FEHLT_BIG2)
)

kategorien_big2 = [
    "pruefkomponente",
    "produkt_spannung",
    "komponente_spannung",
    "komponente_basis",
]
features_big2_neu[kategorien_big2] = (
    features_big2_neu[kategorien_big2].astype(str)
)

features_big2_neu["hauptgruppe"] = (
    faelle_b2["erzeugnis"]
    .map(L2.hauptgruppe_von)
    .astype(str)
)

kategorien_big2.append("hauptgruppe")

assert features_big2_neu["hauptgruppe"].isin(
    HAUPTGRUPPEN_BIG2
).all()



In [0]:
# BIG2 – Zelle 15: Häufigkeitsbaseline P(Menge | Komponente) je Hauptgruppe und Fold
# Generisch: braucht nur Komponente, Menge, Fold. Kein Modell, keine Namen.

import numpy as np
import pandas as pd

f_nb = faelle_b2.copy()
f_nb["erzeugnis"] = f_nb["erzeugnis"].astype(str)
f_nb["komponente"] = f_nb["komponente"].astype(str)
f_nb["hauptgruppe"] = f_nb["erzeugnis"].map(L2.hauptgruppe_von)
assert f_nb["hauptgruppe"].isin(HAUPTGRUPPEN_BIG2).all()
assert f_nb["fold"].notna().all()

f_nb["p_ist_baseline"] = np.nan
f_nb["erwartete_menge_baseline"] = np.nan
f_nb["p_best_baseline"] = np.nan

for hg in HAUPTGRUPPEN_BIG2:
    for fold in sorted(f_nb["fold"].unique()):
        train = f_nb["hauptgruppe"].eq(hg) & f_nb["fold"].ne(fold)
        test = f_nb["hauptgruppe"].eq(hg) & f_nb["fold"].eq(fold)

        # Gewichtete Zählung je Komponente × Menge, ein Produkt zählt insgesamt 1
        z = (
            f_nb.loc[train]
            .groupby(["komponente", "menge_key"])["blockgewicht"].sum()
            .unstack(fill_value=0.0)
        )
        # Laplace: +1 je beobachteter Menge, +1 für „andere Menge“
        n_keys = z.gt(0).sum(axis=1)
        gesamt = z.sum(axis=1) + n_keys + 1.0
        p = (z + z.gt(0).astype(float)).div(gesamt, axis=0)

        t = f_nb.loc[test, ["komponente", "menge_key"]]
        bekannt = t["komponente"].isin(p.index)
        idx = t.index[bekannt]

        zeilen = p.reindex(t.loc[idx, "komponente"].to_numpy())
        keys = t.loc[idx, "menge_key"].to_numpy()
        spalten = pd.Index(p.columns).get_indexer(keys)

        p_ist = np.where(
            spalten >= 0,
            zeilen.to_numpy()[np.arange(len(idx)), np.clip(spalten, 0, None)],
            1.0 / gesamt.reindex(zeilen.index).to_numpy(),  # unbekannte Menge: Rest-Masse
        )
        best_pos = zeilen.to_numpy().argmax(axis=1)
        f_nb.loc[idx, "p_ist_baseline"] = p_ist
        f_nb.loc[idx, "p_best_baseline"] = zeilen.to_numpy()[np.arange(len(idx)), best_pos]
        f_nb.loc[idx, "erwartete_menge_baseline"] = pd.Index(p.columns)[best_pos].astype(float)

f_nb["kandidat_baseline"] = (
    f_nb["p_ist_baseline"].notna()
    & (f_nb["p_best_baseline"] > f_nb["p_ist_baseline"] + 1e-12)
    & (f_nb["erwartete_menge_baseline"] != f_nb["menge_summe"].astype(float))
)
f_nb["auffaelligkeit_baseline"] = np.where(
    f_nb["kandidat_baseline"], 1 - f_nb["p_ist_baseline"], np.nan
)

display(
    f_nb.groupby("hauptgruppe").agg(
        Prüffälle=("komponente", "size"),
        Baseline_bewertbar=("p_ist_baseline", lambda s: int(s.notna().sum())),
        Kandidaten_Baseline=("kandidat_baseline", "sum"),
        LogLoss_Baseline=("p_ist_baseline",
                          lambda s: float(np.mean(-np.log(s.dropna().clip(1e-12, 1))))),
    ).reset_index()
)


In [0]:
# BIG2 – Zelle 16: Nachbarschaft nach Komponentenset (Jaccard), fold-sauber
# Generisch: „Was tragen die k ähnlichsten Produkte an dieser Komponente?“
# Pro Fold werden nur Blöcke aus Trainingsfolds derselben Hauptgruppe als
# Nachbarn zugelassen; das eigene Produkt ist immer ausgeschlossen.

K_NACHBARN = 10

bloecke_nb = (
    f_nb[["erzeugnis", "block", "hauptgruppe", "fold"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
block_idx = pd.MultiIndex.from_frame(bloecke_nb[["erzeugnis", "block"]])
komp_idx = pd.Index(sorted(f_nb["komponente"].unique()))

zeile_je_fall = block_idx.get_indexer(pd.MultiIndex.from_frame(f_nb[["erzeugnis", "block"]]))
spalte_je_fall = komp_idx.get_indexer(f_nb["komponente"])
assert (zeile_je_fall >= 0).all() and (spalte_je_fall >= 0).all()

B = np.zeros((len(bloecke_nb), len(komp_idx)), dtype=np.float32)   # Anwesenheit
Q = np.full((len(bloecke_nb), len(komp_idx)), np.nan, dtype=np.float32)  # Menge
B[zeile_je_fall, spalte_je_fall] = 1.0
Q[zeile_je_fall, spalte_je_fall] = f_nb["menge_summe"].to_numpy(dtype=np.float32)

# Jaccard-Ähnlichkeit zwischen allen Blöcken
schnitt = B @ B.T
groesse = B.sum(axis=1)
vereinigung = groesse[:, None] + groesse[None, :] - schnitt
jaccard = np.where(vereinigung > 0, schnitt / np.maximum(vereinigung, 1e-9), 0.0)

hg_block = bloecke_nb["hauptgruppe"].to_numpy()
fold_block = bloecke_nb["fold"].to_numpy()
erz_block = bloecke_nb["erzeugnis"].to_numpy()

nachbar_merkmale_nb = {}   # fold -> DataFrame (index = f_nb.index)

for fold in sorted(f_nb["fold"].unique()):
    # Zulässige Nachbarn: gleiche Hauptgruppe, nicht im Prüffold, nicht dasselbe Produkt
    sim = jaccard.copy()
    sim[:, fold_block == fold] = -1.0
    sim[hg_block[:, None] != hg_block[None, :]] = -1.0
    sim[erz_block[:, None] == erz_block[None, :]] = -1.0

    top = np.argsort(-sim, axis=1)[:, :K_NACHBARN]                     # (Blöcke × k)
    top_sim = np.take_along_axis(sim, top, axis=1)
    gueltig = top_sim > 0

    nb_zeilen = top[zeile_je_fall]                                     # (Fälle × k)
    nb_gueltig = gueltig[zeile_je_fall]
    nb_menge = Q[nb_zeilen, spalte_je_fall[:, None]]                   # NaN wenn Komponente fehlt
    nb_menge = np.where(nb_gueltig, nb_menge, np.nan)
    hat = ~np.isnan(nb_menge)

    with np.errstate(all="ignore"):
        m = pd.DataFrame(index=f_nb.index)
        m["nb_jaccard_mittel"] = np.where(nb_gueltig, top_sim[zeile_je_fall], np.nan).mean(axis=1)
        m["nb_anteil_mit_komponente"] = hat.sum(axis=1) / np.maximum(nb_gueltig.sum(axis=1), 1)
        m["nb_menge_median"] = np.nanmedian(nb_menge, axis=1)
        m["nb_menge_min"] = np.nanmin(nb_menge, axis=1)
        m["nb_menge_max"] = np.nanmax(nb_menge, axis=1)
        # Diagnose (nutzt eigene Menge, NICHT als Modellmerkmal verwenden)
        ist = f_nb["menge_summe"].to_numpy(dtype=np.float32)[:, None]
        m["nb_anteil_gleiche_menge"] = np.where(
            hat.sum(axis=1) > 0, (hat & (nb_menge == ist)).sum(axis=1) / np.maximum(hat.sum(axis=1), 1), np.nan
        )
    nachbar_merkmale_nb[fold] = m

NB_MODELLMERKMALE = ["nb_jaccard_mittel", "nb_anteil_mit_komponente",
                     "nb_menge_median", "nb_menge_min", "nb_menge_max"]

# Diagnose-Sicht: für jeden Fall die Nachbarn aus seinem eigenen Prüffold
diag = pd.concat(
    [nachbar_merkmale_nb[fold].loc[f_nb["fold"].eq(fold)] for fold in nachbar_merkmale_nb]
).loc[f_nb.index]
f_nb = f_nb.join(diag)

display(
    f_nb.groupby("hauptgruppe").agg(
        Fälle=("komponente", "size"),
        Nachbarn_mit_Komponente_Mittel=("nb_anteil_mit_komponente", "mean"),
        Fälle_ohne_Nachbarmenge=("nb_menge_median", lambda s: int(s.isna().sum())),
        Anteil_gleiche_Menge_Median=("nb_anteil_gleiche_menge", "median"),
    ).reset_index()
)


In [0]:
# Big2: ausschließlich die zehn gespeicherten Nachbar-Modelle laden und auswerten.
# Es gibt hier kein Training. Die Merkmale werden wie beim Snapshot pro Prüffold
# neu gebildet, damit jeder Fold seine eigenen zulässigen Nachbarn verwendet.

import ast
import hashlib
import json
import pickle
from pathlib import Path
from catboost import CatBoostClassifier

SNAPSHOT_NB = Path(
    "/Workspace/Users/khalil-said.albert@de.abb.com/Daten/"
    "big2_nur_nachbarn_2026-09-24_14-51-20_122164"
)
assert SNAPSHOT_NB.is_dir(), f"Big2-Snapshot nicht gefunden: {SNAPSHOT_NB}"
with (SNAPSHOT_NB / "metadata.json").open(encoding="utf-8") as datei:
    metadaten_nb = json.load(datei)
with (SNAPSHOT_NB / "daten.pkl").open("rb") as datei:
    daten_nb = pickle.load(datei)  # Nur den selbst erstellten, vertrauenswürdigen Snapshot laden.

assert metadaten_nb["format"] == 1
assert metadaten_nb["daten_datei"] == "daten.pkl"
assert len(metadaten_nb["modelle"]) == len(HAUPTGRUPPEN_BIG2) * N_FOLDS == 10
assert tuple(daten_nb["referenzdaten"]["HAUPTGRUPPEN_BIG2"]) == tuple(HAUPTGRUPPEN_BIG2)
assert daten_nb["referenzdaten"]["fold_je_produkt"] == fold_je_produkt, (
    "Die Produkt-Folds haben sich seit dem Speichern geändert."
)

# Ein anderer Cleaner oder eine geänderte Stückliste macht alte Vorhersagen
# unzuverlässig. Deshalb zuerst alle Lernantworten und Fold-Zuordnungen prüfen.
schluessel = ["erzeugnis", "block", "komponente"]
referenz = daten_nb["referenzdaten"]["faelle_b2"].copy()

ist_index = pd.MultiIndex.from_frame(faelle_b2[schluessel])
referenz = referenz.set_index(schluessel)
assert ist_index.is_unique and referenz.index.is_unique
assert set(ist_index) == set(referenz.index), "Produkt-/Block-/Komponentenfälle geändert"

faelle_ref_nb = referenz.loc[ist_index].reset_index()
faelle_ref_nb.index = faelle_b2.index

geaendert_nb = (
    faelle_ref_nb["menge_key"].ne(faelle_b2["menge_key"])
    | faelle_ref_nb["anzahl_positionen"].ne(faelle_b2["anzahl_positionen"])
)
print(f"{geaendert_nb.sum()} Mengen-/Positionsfälle verändert")
assert features_big2_neu.index.equals(faelle_b2.index)
assert f_nb.index.equals(faelle_b2.index)

basis_spalten = [c for c in features_big2_neu.columns if c != "hauptgruppe"]
modellspalten_nb = basis_spalten + NB_MODELLMERKMALE
assert len(modellspalten_nb) == len(set(modellspalten_nb)) == 269
assert sorted(f_nb["fold"].astype(int).unique().tolist()) == list(range(N_FOLDS))

# Das gespeicherte X enthält nur die Nachbarwerte des ZULETZT berechneten
# Folds. Es dient als Datenkontrolle, nicht als Eingabe für alle zehn Modelle.
X_letzter_fold = features_big2_neu[basis_spalten].join(
    nachbar_merkmale_nb[N_FOLDS - 1][NB_MODELLMERKMALE]
)[modellspalten_nb]
try:
    pd.testing.assert_frame_equal(
        daten_nb["eingabetabellen"]["X"], X_letzter_fold,
        check_exact=False, rtol=1e-6, atol=1e-6,
        obj="Nachbar-Eingaben gegenüber gespeichertem Snapshot",
    )
    print("✓ Nachbar-Eingaben stimmen mit Snapshot überein.")
except AssertionError as e:
    print(f"⚠ Nachbar-Eingaben weichen vom Snapshot ab (erwartet bei neuen Daten):\n{e}")
del X_letzter_fold

# Neun Modelle besitzen im Snapshot einen ausdrücklichen Fold-Alias. Das
# zehnte ist dort nur 'modell' benannt; die fehlende Zuordnung muss eindeutig
# sein und wird zusätzlich gegen Seed und Trainingsklassen geprüft.
erwartete_schluessel = {
    ("mit_nachbarn", hg, fold)
    for hg in HAUPTGRUPPEN_BIG2 for fold in range(N_FOLDS)
}
infos_nb = {}
ohne_alias_nb = []
for info in metadaten_nb["modelle"]:
    assert info["vorhersage_nach_laden_geprueft"], info["datei"]
    assert info["feature_names"] == modellspalten_nb, (
        f"Merkmalschema passt nicht: {info['datei']}"
    )
    schluessel = set()
    for alias in info["variablennamen"]:
        if alias.startswith("modelle_nb[") and alias.endswith("]"):
            key = ast.literal_eval(alias[len("modelle_nb["):-1])
            schluessel.add(tuple(key))
    assert len(schluessel) <= 1, f"Mehrdeutiger Modell-Alias: {info['datei']}"
    if schluessel:
        key = schluessel.pop()
        assert key in erwartete_schluessel and key not in infos_nb, (
            f"Ungültiger oder doppelter Modell-Alias: {info['datei']} / {key}"
        )
        infos_nb[key] = info
    else:
        ohne_alias_nb.append(info)

fehlende_schluessel_nb = erwartete_schluessel - set(infos_nb)
assert len(fehlende_schluessel_nb) == len(ohne_alias_nb), (
    f"Fold-Zuordnung nicht eindeutig: fehlend={fehlende_schluessel_nb}; "
    f"ohne Alias={[m['datei'] for m in ohne_alias_nb]}"
)
if fehlende_schluessel_nb:
    assert len(fehlende_schluessel_nb) == 1, (
        f"Mehr als ein fehlendes Modell: {fehlende_schluessel_nb}"
    )
    fehlender_schluessel_nb = fehlende_schluessel_nb.pop()
    infos_nb[fehlender_schluessel_nb] = ohne_alias_nb[0]
    print(
        f"Modell ohne Fold-Alias: {ohne_alias_nb[0]['datei']} -> "
        f"{fehlender_schluessel_nb} (Pr\u00fcfung von Seed/Klassen folgt)."
    )
else:
    print("\u2713 Alle Modelle haben explizite Fold-Zuordnungen.")
assert set(infos_nb) == erwartete_schluessel

modelle_nb = {}
ergebnisse_nb = {}
teile_nb = []
for hg in HAUPTGRUPPEN_BIG2:
    for fold in range(N_FOLDS):
        key = ("mit_nachbarn", hg, fold)
        info = infos_nb[key]
        datei = SNAPSHOT_NB / info["datei"]
        assert datei.is_file(), datei
        assert hashlib.sha256(datei.read_bytes()).hexdigest() == info["sha256"], (
            f"Gespeicherte Modelldatei verändert: {datei}"
        )
        modell = CatBoostClassifier()
        modell.load_model(str(datei))
        assert list(map(str, modell.feature_names_)) == modellspalten_nb
        assert list(map(str, modell.classes_)) == info["klassen"]
        assert [modellspalten_nb[i] for i in modell.get_cat_feature_indices()] == info["kategoriale_spalten"]
        assert int(modell.tree_count_) == int(info["baeume"])
        assert int(modell.get_all_params()["random_seed"]) == SEED + fold, (
            f"Seed/Fold passen nicht: {info['datei']} / {key}"
        )

        in_hg = f_nb["hauptgruppe"].eq(hg)
        train = in_hg & f_nb["fold"].ne(fold)
        test = in_hg & f_nb["fold"].eq(fold)
        assert train.any() and test.any()
        assert set(f_nb.loc[train, "erzeugnis"]).isdisjoint(
            set(f_nb.loc[test, "erzeugnis"])
        )
        assert set(map(str, modell.classes_)) == set(faelle_b2.loc[train, "zustand_big2"]), (
            f"Trainingsklassen passen nicht: {info['datei']} / {key}"
        )

        # Für diesen Fold nur die gespeicherten Modelle verwenden; alle fünf
        # Nachbarschaftswerte stammen aus den anderen vier Folds derselben HG.
        X_fold = features_big2_neu[basis_spalten].join(
            nachbar_merkmale_nb[fold][NB_MODELLMERKMALE]
        )[modellspalten_nb]
        erg = prognose_big2(
            modell, faelle_b2.loc[train], faelle_b2.loc[test], X_fold.loc[test]
        )
        erg.index = faelle_b2.index[test]
        teile_nb.append(erg)
        modelle_nb[key] = modell
        print(f"Geladen: {hg} | Fold {fold} | {int(test.sum())} Prüffälle | {info['datei']}")

e = pd.concat(teile_nb).loc[f_nb.index]
e["hauptgruppe"] = f_nb["hauptgruppe"]
ergebnisse_nb["mit_nachbarn"] = e
assert len(modelle_nb) == 10 and len(e) == len(f_nb)

vergleich_nb = []
for hg in HAUPTGRUPPEN_BIG2:
    in_hg = f_nb["hauptgruppe"].eq(hg)
    gemeinsam = in_hg & f_nb["p_ist_baseline"].notna() & e["p_ist_menge"].notna()
    w = f_nb.loc[gemeinsam, "blockgewicht"]
    assert gemeinsam.any()
    vergleich_nb.append({
        "Hauptgruppe": hg,
        "Gemeinsam bewertbar": int(gemeinsam.sum()),
        "LogLoss Baseline": np.average(
            -np.log(f_nb.loc[gemeinsam, "p_ist_baseline"].clip(1e-12, 1)), weights=w
        ),
        "Kandidaten Baseline": int(f_nb.loc[in_hg, "kandidat_baseline"].sum()),
        "LogLoss mit_nachbarn": np.average(
            -np.log(e.loc[gemeinsam, "p_ist_menge"].clip(1e-12, 1)), weights=w
        ),
        "Kandidaten mit_nachbarn": int(e.loc[in_hg, "kandidat_big2"].sum()),
        "Bewertbar mit_nachbarn": int(e.loc[in_hg, "p_ist_menge"].notna().sum()),
    })
display(pd.DataFrame(vergleich_nb).round(5))


In [0]:
# BIG2 – Zelle 18 (Anzeige neu): Produktaggregation im Kandidatenlisten-Format
# Eine Zeile je Kandidat, gruppiert nach Produkt; Produktbefund und Produktrang stehen mit dabei.

MIN_KANDIDATEN_MUSTER = 3

def produkt_aggregation(e, name):
    k = e.loc[e["kandidat_big2"] & e["auffaelligkeit_big2"].notna()].copy()
    k["neglogp"] = -np.log(k["p_ist_menge"].clip(1e-12, 1))
    k["verhaeltnis"] = (k["beobachtete_menge"] / k["erwartete_menge"]).round(3)
    agg = (
        k.groupby(["hauptgruppe", "erzeugnis", "block"])
        .agg(Kandidaten=("komponente", "size"),
             Summe_neglogp=("neglogp", "sum"),
             Verhaeltnisse=("verhaeltnis", lambda s: sorted(s.unique())))
        .reset_index()
    )
    konstant = agg["Verhaeltnisse"].map(len).eq(1) & (agg["Kandidaten"] >= MIN_KANDIDATEN_MUSTER)
    agg["Produktbefund"] = np.where(
        konstant,
        agg["Verhaeltnisse"].map(lambda v: f"Alle Mengen um Faktor {v[0]:g} verschoben"),
        np.where(agg["Kandidaten"] >= MIN_KANDIDATEN_MUSTER,
                 "Mehrere unabhängige Abweichungen", "Einzelabweichung"),
    )
    agg["Variante"] = name
    agg = agg.sort_values("Summe_neglogp", ascending=False)
    agg["Rang_Produkt_in_HG"] = agg.groupby("hauptgruppe").cumcount() + 1
    return agg.drop(columns="Verhaeltnisse")


def kandidatenliste_produktweise(e, agg, top_n_produkte=10):
    k = e.loc[e["kandidat_big2"] & e["auffaelligkeit_big2"].notna()].copy()
    k = k.merge(
        agg[["hauptgruppe", "erzeugnis", "block", "Rang_Produkt_in_HG", "Kandidaten", "Produktbefund"]],
        on=["hauptgruppe", "erzeugnis", "block"], how="inner",
    )
    k = k.loc[k["Rang_Produkt_in_HG"] <= top_n_produkte]
    k = k.sort_values(["hauptgruppe", "Rang_Produkt_in_HG", "auffaelligkeit_big2"],
                      ascending=[True, True, False])
    return k[[
        "hauptgruppe", "Rang_Produkt_in_HG", "erzeugnis", "erzeugnis_txt",
        "komponente", "komponente_txt", "beobachtete_menge", "erwartete_menge",
        "grund_big2", "auffaelligkeit_big2", "Kandidaten", "Produktbefund",
    ]].rename(columns={
        "hauptgruppe": "Hauptgruppe", "Rang_Produkt_in_HG": "Rang Produkt",
        "erzeugnis": "Identnummer", "erzeugnis_txt": "Materialkurztext",
        "komponente": "Identnummer Teil", "komponente_txt": "Materialkurztext Teil",
        "beobachtete_menge": "Menge aktuell", "erwartete_menge": "Menge erwartet",
        "grund_big2": "Grund", "auffaelligkeit_big2": "Auffälligkeit Big2",
        "Kandidaten": "Kandidaten im Produkt", "Produktbefund": "Produktbefund",
    })


namen_nb = pd.DataFrame(sauber)
namen_nb["erzeugnis"] = namen_nb["erzeugnis"].astype(str)
namen_nb["komponente"] = namen_nb["komponente"].astype(str)
p_txt = namen_nb.drop_duplicates("erzeugnis").set_index("erzeugnis")["erzeugnis_txt"]
k_txt = namen_nb.drop_duplicates("komponente").set_index("komponente")["komponente_txt"]

aggregationen_nb = {}
for v, e in ergebnisse_nb.items():
    e = e.copy()
    e["erzeugnis"] = e["erzeugnis"].astype(str)
    e["komponente"] = e["komponente"].astype(str)
    e["erzeugnis_txt"] = e["erzeugnis"].map(p_txt)
    e["komponente_txt"] = e["komponente"].map(k_txt).astype(str).str.strip()
    e["Rang_Zeile_in_HG"] = (
        e.loc[e["kandidat_big2"]].groupby("hauptgruppe")["auffaelligkeit_big2"]
        .rank(ascending=False, method="first")
    )
    ergebnisse_nb[v] = e
    aggregationen_nb[v] = produkt_aggregation(e, v)

    liste = kandidatenliste_produktweise(e, aggregationen_nb[v], top_n_produkte=10)
    for hg in HAUPTGRUPPEN_BIG2:
        teil = liste.loc[liste["Hauptgruppe"].eq(hg)].drop(columns="Hauptgruppe")
        print(f"=== {v} | {hg}: Top 10 Produkte, {len(teil)} Kandidatenzeilen ===")
        display(teil.round(4))
